# Event Hub to Landing Zone

## Overview
Streaming ingestion pipeline reading device telemetry from Azure Event Hub and writing to the landing zone (bronze layer).

**Source**: Azure Event Hub (device telemetry stream)
**Target**: Landing zone (JSON format)
**Pattern**: Raw payload preservation with ingestion metadata

---

## Step 1: Event Hub Configuration
Import dependencies and configure Event Hub connection string.

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

connectionString = "****************************"

event_hub_conf = {
  'eventhubs.connectionString': connectionString
}

---
## Step 2: Read Stream from Event Hub
Create streaming DataFrame reading from Event Hub source.

In [0]:
raw_stream_df = (
    spark.readStream
    .format("eventhubs")
    .options(**event_hub_conf)
    .load()
)

---
## Step 3: Transform Raw Payload
Extract raw JSON payload and add ingestion metadata (timestamp, source system, partition date).

In [0]:
%python
#Extract Raw JSON Payload
bronze_df = (
    raw_stream_df
    .select(
        col("body").cast("string").alias("raw_payload")
    )
    .withColumn("ingestion_time", current_timestamp())
    .withColumn("source_system", lit("eventhub"))
    .withColumn("partition_date", current_date())
)

---
## Step 4: Write Stream to Landing Zone
Write streaming data to landing zone in JSON format with checkpointing for fault tolerance.

In [0]:
(
    bronze_df.writeStream
    .format("json")
    .outputMode("append")
    .option(
        "checkpointLocation",
        "abfss://landing@mytelecomstorage.dfs.core.windows.net/Device_logs/checkpoints"
    )
    .start(
        "abfss://landing@mytelecomstorage.dfs.core.windows.net/Device_logs"
    )
)

---
## Summary

**Pipeline**: Real-time streaming ingestion from Azure Event Hub to landing zone

**Data Flow**:
1. Read stream from Event Hub (eventhubs format)
2. Extract raw JSON payload from body field
3. Add metadata: ingestion_time, source_system, partition_date
4. Write to landing zone (JSON append mode with checkpointing)

**Output Schema**:
* raw_payload (string) - Original JSON from Event Hub
* ingestion_time (timestamp) - Record ingestion timestamp
* source_system (string) - Source identifier ("eventhub")
* partition_date (date) - Date partition for organization

**Target**: abfss://landing@mytelecomstorage.dfs.core.windows.net/Device_logs